In [6]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#


$$
f: (0, 1)^n \times \{0, 1\}^n \to \mathbb{R}, \quad y = 
L(\mathbf{p}, \mathbf{t}) = 
-\frac{1}{N} \sum_i t_i \, \ln(p_i) 
$$

$$ \sum_i p_i = 1, \quad \sum_i t_i = 1 $$

$$ \frac{\partial f}{\partial p_i} = - \frac{1}{N} \, t_i \frac{1}{p_i} $$

$$ 
\frac{\partial f}{\partial \mathbf{p}} = 
-\frac{1}{N} \, \mathbf{t} \oslash \mathbf{p} 
$$

$$
J_f(\mathbf{p}) =
\nabla _f(\mathbf{p}) \top =
\begin{bmatrix}
    \frac{\partial f}{\partial p_1}, &
    \frac{\partial f}{\partial p_2},  &
    \cdots, &
    \frac{\partial f}{\partial p_n}
\end{bmatrix}
$$

$$ df = J_f(\mathbf{p}) \cdot d\mathbf{p} $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{df}, df \right \rangle =
\left \langle \frac{dF}{d\mathbf{p}}, d\mathbf{p} \right \rangle
$$

$$
\left \langle \frac{dF}{df}, J_f(\mathbf{p}) \cdot d\mathbf{p} \right \rangle =
\left \langle \frac{dF}{d\mathbf{p}}, d\mathbf{p} \right \rangle \implies
$$

$$ \frac{dF}{d\mathbf{p}} = J_f(\mathbf{p})^\top \, \frac{dF}{df} $$

$$ \frac{dF}{d\mathbf{p}} = \frac{df}{d\mathbf{p}} \odot \frac{dF}{df} $$

In [11]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import common # type: ignore
from approx import approx # type: ignore

def _ce(p: tr.Tensor, t: tr.Tensor, reduction='mean') -> tr.Tensor:
    """
    Computes the cross-entropy loss between predictions `p` and targets `t` with `mean` reduction.
    Note: This function assumes that `p` has already been clamped to avoid log(0) issues.
    """

    if reduction != 'mean':
        assert False, f"Unsupported reduction: {reduction}"

    return -(t * tr.log(p)).mean()


def ce(p: tr.Tensor, t: tr.Tensor, reduction='mean') -> tr.Tensor:
    """
    Computes the cross-entropy loss between predictions `p` and targets `t` with `mean` reduction.
    Note: This function clamps `p` to avoid log(0) issues.
    """

    assert tr.all((p >= -1.0) & (p <= +1.0))

    eps = 1e-7
    pc = p.clamp(eps, 1 - eps)
    L = _ce(pc, t, reduction)
    return L


class CEFunction(autograd.Function):
    """Custom autograd function for the `cross-entropy`."""

    @staticmethod
    def forward(ctx, p, t):
        eps = 1e-7
        pc = p.clamp(eps, 1 - eps)
        ctx.save_for_backward(pc, t)
        y = _ce(pc, t)
        return y

    @staticmethod
    def backward(ctx, dF_df):
        (pc, t) = ctx.saved_tensors
        S = pc.shape[0]
        FO = pc.shape[1]
        N = S * FO

        df_dp = (-1 / N) * (t / pc)
        dF_dp = df_dp * dF_df

        return (dF_dp, None)
    
class CE(nn.Module):
    """Cross-entropy loss module."""

    def __init__(self, reduction='mean'):
        super().__init__()

        if reduction != "mean":
            assert False, f"Unsupported reduction: {reduction}"

    def forward(self, p: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        return CEFunction.apply(p, t)

In [20]:
# Unit tests

import torch


def test_gradcheck(samples=10):
    def fn(p, t):
        return CEFunction.apply(p, t)

    p = tr.rand(samples, 1, dtype=tr.float64, requires_grad=True)
    t = tr.randint(0, 5, (samples, 1)).float().requires_grad_(False)
    assert autograd.gradcheck(fn, (p, t), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(samples):
    p = tr.rand(samples, 1, requires_grad=True)
    t = tr.randint(0, 5, (samples, 1)).float()

    p1 = p.clone().detach().requires_grad_(True)
    y1 = CE(reduction='mean')(tr.softmax(p1, dim=1), t)
    y1.backward()

    p2 = p.clone().detach().requires_grad_(True)
    y2 = nn.CrossEntropyLoss(reduction='mean')(p2, t)
    y2.backward()

    assert y1.item() == approx(y2.item()).log()
    assert p1.grad == approx(p2.grad)


if __name__ == "__main__":
    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(3)
    test_compare(10)
    test_compare(100)